In [ ]:
!pip install torch-sparse
%restart_python

In [39]:
# Mitigates CUDA fragmentation OOMs on deep K-sweep configs. Must run
# before any CUDA context is created (before `import torch` elsewhere).
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


## Setup

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
from torch_geometric.utils import degree

from training_data import Ontology, TrainingData
from node_prediction.gnn_tools import (
    parse_neighbors,
    get_graph_mask,
    train_final_model,
    evaluate_test_set,
    compute_entity_reuse_stats,
    avg_children_per_case,
    train_model,
    evaluate_ablation_configuration,
    run_ablation_run,
    run_sequential_optimization_with_verification,
    run_k_sweep,
)

In [150]:
FEATURE_DIM = 1024
HIDDEN_DIM = 64
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"

sulo_schema_path = f"{volume}sulo_kg.ttl"
sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"
sulo_vars = f'{volume}sulo_vars.pt'

baseline_schema_path = f"{volume}base_kg.ttl"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"
baseline_vars = f'{volume}base_vars.pt'

## Create Training Data and Graph

In [ ]:
spark.sql("""
    SELECT case_id FROM ds_ccc.kg_sulo_balanced_case_ids
""").createOrReplaceTempView("sampled_case_ids")

embedding_table = spark.sql("""
    SELECT 
        e.*
    FROM sampled_case_ids c
    JOIN ds_ccc.sulokg_embeddings_dv e ON c.case_id = e.case_id
""")

task_embedding_table = spark.sql("""
    SELECT 
        e.*
    FROM sampled_case_ids c
    JOIN ds_ccc.sulokg_task_embeddings_dv e ON c.case_id = e.case_id
""")

task_embedding_df = task_embedding_table.toPandas()
embedding_df = embedding_table.toPandas()

In [152]:
def cap_max_degree(edge_index, edge_type, num_nodes, max_degree=128, seed=42):
    """
    Caps in-degree per node at max_degree by randomly keeping at most
    max_degree incoming edges for any node whose in-degree exceeds it.
    Applied to the combined (already-bidirectional) edge_index, so this
    bounds fan-in from both directions -- exactly what NeighborLoader's
    per-hop sampling actually pulls from.
    """
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(edge_index.size(1), generator=g)
    edge_index = edge_index[:, perm]
    edge_type = edge_type[perm]

    dst = edge_index[1]
    order = torch.argsort(dst, stable=True)  # stable sort preserves the random shuffle within each dst group
    dst_sorted = dst[order]

    counts = torch.bincount(dst_sorted, minlength=num_nodes)
    group_start = torch.zeros(num_nodes, dtype=torch.long)
    group_start[1:] = torch.cumsum(counts[:-1], dim=0)
    pos_in_group = torch.arange(dst_sorted.size(0)) - group_start[dst_sorted]

    keep_mask_sorted = pos_in_group < max_degree
    keep_order = order[keep_mask_sorted]

    print(f"Capped edges: {edge_index.size(1)} -> {keep_order.size(0)} "
          f"({edge_index.size(1) - keep_order.size(0)} dropped, max_degree={max_degree})")
    return edge_index[:, keep_order], edge_type[keep_order]

def compute_degree_feature(edge_index, num_nodes):
    """Log-scaled, z-normalized in-degree -- lets the model tell hub nodes
    apart even when they share an identical categorical fallback vector."""
    deg = degree(edge_index[1], num_nodes=num_nodes).float()
    log_deg = torch.log1p(deg)
    normalized = (log_deg - log_deg.mean()) / (log_deg.std() + 1e-8)
    return normalized

In [ ]:
base_graph = Ontology(
    nt_file_path=baseline_data_path,
    label_predicate="hasSurveyRating",
    label_object_pattern=r"SurveyRating/(\d+)",
)
base_training_data = TrainingData()
base_training_data.create_training_data(base_graph)

base_vector_table, base_node_index = base_graph.build_feature_lookup(
    emb_dim=FEATURE_DIM, embedding_df=embedding_df, task_embedding_df=task_embedding_df,
)

base_y = torch.zeros(base_graph.num_nodes, dtype=torch.float)
base_y[torch.cat([base_training_data.x_train, base_training_data.x_val, base_training_data.x_test])] = \
    torch.cat([base_training_data.y_train, base_training_data.y_val, base_training_data.y_test])

base_degree_feat = compute_degree_feature(base_graph.edge_index, base_graph.num_nodes)

base_edge_index, base_edge_type = cap_max_degree(
    base_graph.edge_index.contiguous(), base_graph.edge_type.contiguous(),
    num_nodes=base_graph.num_nodes, max_degree=128,
)

base_graph_data = Data(
    edge_index=base_edge_index,
    edge_type=base_edge_type,
    y=base_y,
    num_nodes=base_graph.num_nodes,
    n_id_to_vec_row=base_node_index,
    degree_feat=base_degree_feat,
)

torch.save({
    'base_graph_data': base_graph_data,
    'x_train': base_training_data.x_train,
    'x_val': base_training_data.x_val,
    'x_test': base_training_data.x_test,
    'base_vector_table': base_vector_table
}, baseline_vars)

[DRIVER_STOPPED] Driver down cause: driver state change (exit code: 137)

Error: [DRIVER_STOPPED] Driver down cause: driver state change (exit code: 137)

In [154]:
sulo_graph = Ontology(
    nt_file_path=sulo_data_path,
    label_predicate="hasFeature",
    label_object_pattern=r"SurveyRating/(\d+)",
)
sulo_training_data = TrainingData()
sulo_training_data.create_training_data(sulo_graph)

sulo_vector_table, sulo_node_index = sulo_graph.build_feature_lookup(
    emb_dim=FEATURE_DIM, embedding_df=embedding_df, task_embedding_df=task_embedding_df,
)

sulo_y = torch.zeros(sulo_graph.num_nodes, dtype=torch.float)
sulo_y[torch.cat([sulo_training_data.x_train, sulo_training_data.x_val, sulo_training_data.x_test])] = \
    torch.cat([sulo_training_data.y_train, sulo_training_data.y_val, sulo_training_data.y_test])

sulo_degree_feat = compute_degree_feature(sulo_graph.edge_index, sulo_graph.num_nodes)

sulo_edge_index, sulo_edge_type = cap_max_degree(
    sulo_graph.edge_index.contiguous(), sulo_graph.edge_type.contiguous(),
    num_nodes=sulo_graph.num_nodes, max_degree=128,
)

sulo_graph_data = Data(
    edge_index=sulo_edge_index,
    edge_type=sulo_edge_type,
    y=sulo_y,
    num_nodes=sulo_graph.num_nodes,
    n_id_to_vec_row=sulo_node_index,
    degree_feat=sulo_degree_feat,
)

torch.save({
    'sulo_graph_data': sulo_graph_data,
    'x_train': sulo_training_data.x_train,
    'x_val': sulo_training_data.x_val,
    'x_test': sulo_training_data.x_test,
    'sulo_vector_table': sulo_vector_table
}, sulo_vars)

ContextNotFound:

Error: ContextNotFound: 

## Intialize Model and Training

In [ ]:
class RGCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels_list, num_relations, 
                 vector_table, categorical_row_ids, num_bases=None, 
                 out_channels=1, dropout=0.2):
        super().__init__()
        num_rows, emb_dim = vector_table.shape

        is_categorical = torch.zeros(num_rows, dtype=torch.bool)
        is_categorical[categorical_row_ids] = True
        self.register_buffer('is_categorical_mask', is_categorical)

        # Plain attribute, not a registered buffer -- avoids model.to()/.cpu()
        # cloning this multi-GB table on every call. Caller must move it to
        # device once, outside the sweep loop, before the first model_factory().
        self.frozen_table = vector_table

        self.categorical_embedding = nn.Embedding(len(categorical_row_ids), emb_dim)
        with torch.no_grad():
            self.categorical_embedding.weight.copy_(
                vector_table[categorical_row_ids.to(vector_table.device)]
            )

        local_id = torch.full((num_rows,), -1, dtype=torch.long)
        local_id[categorical_row_ids] = torch.arange(len(categorical_row_ids))
        self.register_buffer('local_categorical_id', local_id)

        self.convs = nn.ModuleList()
        dims = [in_channels] + hidden_channels_list
        for i in range(len(dims) - 1):
            self.convs.append(RGCNConv(dims[i], dims[i + 1], num_relations=num_relations, num_bases=num_bases))
        self.convs.append(RGCNConv(dims[-1], out_channels, num_relations=num_relations, num_bases=num_bases))
        self.dropout = dropout

    def forward(self, node_row_ids, edge_index, edge_type, degree_feat):
        x = self.frozen_table[node_row_ids].clone()

        cat_mask = self.is_categorical_mask[node_row_ids]
        if cat_mask.any():
            local_ids = self.local_categorical_id[node_row_ids[cat_mask]]
            x[cat_mask] = self.categorical_embedding(local_ids)

        x = torch.cat([x, degree_feat.unsqueeze(1)], dim=1)

        for conv in self.convs[:-1]:
            x = conv(x, edge_index, edge_type)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index, edge_type)
        return x

## Hyperparameter Tuning

In [ ]:
# ── 1. CONFIGURATION & DIRECTORIES ───────────────────────────────────────────
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
os.makedirs(f"{volume}rgcn/tuning_logs/", exist_ok=True)

# ── 1. CONFIGURATION & GRIDS ─────────────────────────────────────────────────
GRID_1A_DIM = [32, 64, 128, 256]

GRID_1B_DEPTH = [
    (1, (25,)),
    (2, (25, 10)),
    (3, (15, 10, 5))
]

GRID_2A_BASES = [None, 15, 30, 45, 60]
GRID_2B_LR = [1e-4, 5e-4, 1e-3, 5e-3]
GRID_2C_WD = [1e-5, 1e-4, 5e-4, 1e-3]

EPOCHS = 50
PATIENCE = 15
DEFAULT_BASES = 30
DEFAULT_LR = 1e-3
DEFAULT_WD = 1e-4

In [ ]:
base_variables = torch.load(baseline_vars, weights_only=False)

base_graph_data = base_variables['base_graph_data']
base_training_data = base_variables['x_train']
base_val_data = base_variables['x_val']
base_test_data = base_variables['x_test']
base_vector_table = base_variables['base_vector_table']

# Move the frozen feature table to the training device exactly once, outside
# the sweep -- RGCNModel now keeps a plain reference to it (see cell above)
# instead of cloning it per model_factory() call, so it must already live on
# the right device before the first model is constructed.
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_vector_table = base_vector_table.to(_device)

num_base_relations = base_graph_data.edge_type.max().item() + 1
node_index = base_graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=base_vector_table.shape[0])
base_categorical_ids = torch.where(usage_counts > 1)[0]

def base_model_factory(hidden_list, bases):
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_base_relations,
        vector_table=base_vector_table,
        categorical_row_ids=base_categorical_ids,
        num_bases=bases,
    )

base_tuning_df = run_sequential_optimization_with_verification(
    model_factory=base_model_factory,
    graph_data=base_graph_data,
    train_indices=base_training_data,
    val_indices=base_val_data,
    test_indices=base_test_data,
    vector_table=base_vector_table,
    schema_label='base',
    save_dir=f"{volume}rgcn/",
    csv_path=f"{volume}rgcn/base_sequential_tuning_results.csv",
    grid_dim=GRID_1A_DIM, grid_depth=GRID_1B_DEPTH, grid_bases=GRID_2A_BASES,
    grid_lr=GRID_2B_LR, grid_wd=GRID_2C_WD,
    default_bases=DEFAULT_BASES, default_lr=DEFAULT_LR, default_wd=DEFAULT_WD,
    epochs=EPOCHS, patience=PATIENCE,
)

In [ ]:
sulo_variables = torch.load(sulo_vars, weights_only=False)

sulo_graph_data = sulo_variables['sulo_graph_data']
sulo_training_data = sulo_variables['x_train']
sulo_val_data = sulo_variables['x_val']
sulo_test_data = sulo_variables['x_test']
sulo_vector_table = sulo_variables['sulo_vector_table']

# Move the frozen feature table to the training device exactly once, outside
# the sweep -- see the matching comment in the base cell above.
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
sulo_vector_table = sulo_vector_table.to(_device)

num_sulo_relations = sulo_graph_data.edge_type.max().item() + 1
node_index = sulo_graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=sulo_vector_table.shape[0])
sulo_categorical_ids = torch.where(usage_counts > 1)[0]

def sulo_model_factory(hidden_list, bases):
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_sulo_relations,
        vector_table=sulo_vector_table,
        categorical_row_ids=sulo_categorical_ids,
        num_bases=bases,
    )

sulo_tuning_df = run_sequential_optimization_with_verification(
    model_factory=sulo_model_factory,
    graph_data=sulo_graph_data,
    train_indices=sulo_training_data,
    val_indices=sulo_val_data,
    test_indices=sulo_test_data,
    vector_table=sulo_vector_table,
    schema_label='sulo',
    save_dir=f"{volume}rgcn/",
    csv_path=f"{volume}rgcn/sulo_sequential_tuning_results.csv",
    grid_dim=GRID_1A_DIM, grid_depth=GRID_1B_DEPTH, grid_bases=GRID_2A_BASES,
    grid_lr=GRID_2B_LR, grid_wd=GRID_2C_WD,
    default_bases=DEFAULT_BASES, default_lr=DEFAULT_LR, default_wd=DEFAULT_WD,
    epochs=EPOCHS, patience=PATIENCE,
)

In [ ]:
schema_label = 'base'

csv_path = f"{volume}rgcn/{schema_label}_sequential_tuning_results.csv"
ablation_csv_path = f"{volume}rgcn/{schema_label}_ablation_study_results.csv"

# ── 2. RETRIEVE METRIC & ARCHITECTURE WINNERS FROM SEQUENTIAL SWEEP ──────────
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Could not find tuning CSV results at: {csv_path}")

df_tuning = pd.read_csv(csv_path)

phase3_runs = df_tuning[(df_tuning['status'] == 'ok') & (df_tuning['phase'].isin(['3_Verification', '3_Verification_Cached']))]
if len(phase3_runs) == 0:
    phase3_runs = df_tuning[df_tuning['status'] == 'ok']
best_row = phase3_runs.loc[phase3_runs['val_roc_auc'].idxmax()]

optimal_dim = int(best_row['hidden_dim'])
optimal_bases = best_row['num_bases']
optimal_bases = None if pd.isna(optimal_bases) or optimal_bases == 'None' else int(float(optimal_bases))
optimal_lr = float(best_row['lr'])
optimal_wd = float(best_row['weight_decay'])
best_depth = int(best_row['layers'])
optimal_neighbors = parse_neighbors(best_row['neighbors'])

print(f">>> Dynamic Hyperparameter Handoff Complete for {schema_label.upper()}:")
print(f"  * Hidden Dim: {optimal_dim} | Depth: {best_depth} | Bases: {optimal_bases} | LR: {optimal_lr} | WD: {optimal_wd}")

# ── 3. SELF-HEALING ONTOLOGY LOOKUP HELPERS ──────────────────────────────────
vars_path = f"{volume}{schema_label}_vars.pt"
ckpt = torch.load(vars_path, map_location='cpu', weights_only=False)

graph_data = ckpt[f'{schema_label}_graph_data']
train_indices = ckpt['x_train']
val_indices = ckpt['x_val']
test_indices = ckpt['x_test']
vector_table = ckpt[f'{schema_label}_vector_table']

# Move the frozen feature table to the training device exactly once, outside
# the ablation/K-sweep loops below -- see the matching comment in RGCNModel.
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
vector_table = vector_table.to(_device)

num_relations = graph_data.edge_type.max().item() + 1
node_index = graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])

# Cache-first: avoids a fresh Ontology(.nt) parse once
# {schema_label}_ontology_enums.pt exists (written by the save cell right after this one).
_enum_cache_path = f"{volume}{schema_label}_ontology_enums.pt"
if os.path.exists(_enum_cache_path):
    from types import SimpleNamespace
    _enum_cache = torch.load(_enum_cache_path, map_location='cpu', weights_only=False)
    ontology_obj = SimpleNamespace(
        enum_nodes=_enum_cache['enum_nodes'],
        enum_relations=_enum_cache['enum_relations'],
    )
    print(f"[{schema_label}] Loaded cached enum_nodes/enum_relations from {_enum_cache_path} "
          f"-- no .nt file access needed.")
elif schema_label == 'sulo':
    ontology_obj = Ontology(nt_file_path=sulo_data_path, label_predicate="hasFeature", label_object_pattern=r"SurveyRating/(\d+)")
else:
    ontology_obj = Ontology(nt_file_path=baseline_data_path, label_predicate="hasSurveyRating", label_object_pattern=r"SurveyRating/(\d+)")

## Cache ontology enums (enum_nodes / enum_relations) for future .nt-free runs

In [ ]:
from types import SimpleNamespace as _SimpleNamespace
if isinstance(ontology_obj, _SimpleNamespace):
    print(f"[{schema_label}] ontology_obj is already the cached stand-in -- nothing new to save.")
else:
    _enum_cache_path = f"{volume}{schema_label}_ontology_enums.pt"
    torch.save({
        'enum_nodes': ontology_obj.enum_nodes,
        'enum_relations': ontology_obj.enum_relations,
    }, _enum_cache_path)
    print(f"Saved: {_enum_cache_path}")

## Feature Ablation

In [ ]:

# ── 5. RUN EXECUTOR & LOGGING ────────────────────────────────────────────────
ablation_results = []
baseline_test_auc = None
baseline_test_f1 = None

def model_factory():
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=[optimal_dim] * (best_depth - 1),
        num_relations=num_relations,
        vector_table=vector_table,
        categorical_row_ids=torch.where(usage_counts > 1)[0],
        num_bases=optimal_bases,
    )

def log_ablation_run(name, test_data, pct_removed):
    global baseline_test_auc, baseline_test_f1
    record, baseline_test_auc, baseline_test_f1 = run_ablation_run(
        name, test_data, pct_removed, model_factory, baseline_test_auc, baseline_test_f1,
        graph_data=graph_data, schema_label=schema_label,
        train_indices=train_indices, val_indices=val_indices, test_indices=test_indices,
        vector_table=vector_table, usage_counts=usage_counts,
        neighbors=optimal_neighbors, lr=optimal_lr, wd=optimal_wd,
        epochs=50, patience=15,
    )
    ablation_results.append(record)
    pd.DataFrame(ablation_results).to_csv(ablation_csv_path, index=False)

# ── 6. EXECUTE BASELINE CONTROL RUN ──────────────────────────────────────────
log_ablation_run("Baseline Control", graph_data, 0.0)

# ── 6.5. ENTITY FREQUENCY / REUSE STATISTICS ─────────────────────────────────
print("\n>>> COMPUTING ENTITY FREQUENCY / REUSE STATISTICS")

target_concepts = ['CaseDescription', 'TaskDescription', 'CaseSolutionFinalStatement', 'Vehicle', 'Account', 'Task']

entity_stats_df, num_cases, case_mask = compute_entity_reuse_stats(
    graph_data, ontology_obj, schema_label, target_concepts
)

vehicle_mask = get_graph_mask(graph_data, ontology_obj, schema_label, 'Vehicle')
task_mask = get_graph_mask(graph_data, ontology_obj, schema_label, 'Task')
avg_vehicles_per_case = avg_children_per_case(graph_data, case_mask, vehicle_mask, num_cases)
avg_tasks_per_case = avg_children_per_case(graph_data, case_mask, task_mask, num_cases)

print(f"\nTotal Case nodes: {num_cases}")
print(entity_stats_df.to_string(index=False))
print(f"\nAverage Vehicles per Case: {avg_vehicles_per_case:.4f}")
print(f"Average Tasks per Case: {avg_tasks_per_case:.4f}")

entity_stats_path = f"{volume}rgcn/{schema_label}_entity_reuse_stats.csv"
entity_stats_df.to_csv(entity_stats_path, index=False)
print(f"\nSaved entity reuse stats to: {entity_stats_path}")

# ── 7. PASS 1: SINGLE-CONCEPT ABLATION SWEEPS ───────────────────────────────
print("\n>>> STARTING PASS 1: SINGLE-CONCEPT ABLATION SWEEP")

for concept in target_concepts:
    concept_mask = get_graph_mask(graph_data, ontology_obj, schema_label, concept)
    if concept_mask.sum().item() == 0:
        print(f"  -> Skipping '{concept}': no matching nodes found.")
        continue

    edge_mask = ~(concept_mask[graph_data.edge_index[0]] | concept_mask[graph_data.edge_index[1]])
    masked_data = graph_data.clone()
    masked_data.edge_index = graph_data.edge_index[:, edge_mask]
    masked_data.edge_type = graph_data.edge_type[edge_mask]

    pct_removed = ((graph_data.edge_index.size(1) - masked_data.edge_index.size(1)) / graph_data.edge_index.size(1)) * 100.0
    if pct_removed > 0.0:
        log_ablation_run(f"Pruned Concept: {concept}", masked_data, pct_removed)

# ── 8. PASS 2: CORE VS. STRUCTURE GROUP ABLATIONS ────────────────────────────
print("\n>>> STARTING PASS 2: GROUP CORE ABLATIONS")

# -- Group A: Semantic Core (Pruning structural categoricals, keeping Case -> Task -> Text)
semantic_data = graph_data.clone()
cat_masks = torch.zeros(graph_data.num_nodes, dtype=torch.bool)

for cls in [
    'Account', 'AccountType', 'Brand', 'CaseClosureType', 'CaseModification', 
    'CaseReason', 'CaseStatus', 'CaseType', 'Country', 'Language', 
    'PriorityLevel', 'ProductGroup', 'ServiceType', 'SupportLevel', 
    'TaskType', 'TimeInstant', 'Vehicle', 'VehicleClass', 'VehicleSeries', 
    'hasAccountLastModifiedDate', 'hasCaseClosedDate', 'hasCaseConversationAmount', 
    'hasCaseHandlingTimeSec', 'hasCaseSessionAmount', 'hasCreateDate', 
    'hasModificationDate', 'hasTaskCompletedDate', 'hasTaskDurationSec'
]:
    cat_masks |= get_graph_mask(graph_data, ontology_obj, schema_label, cls)

mask = ~(cat_masks[graph_data.edge_index[0]] | cat_masks[graph_data.edge_index[1]])
semantic_data.edge_index = graph_data.edge_index[:, mask]
semantic_data.edge_type = graph_data.edge_type[mask]

pct_removed = ((graph_data.edge_index.size(1) - semantic_data.edge_index.size(1)) / graph_data.edge_index.size(1)) * 100.0
log_ablation_run("Semantic Core (Paths Only)", semantic_data, pct_removed)

# -- Group B: Structural Core (Pruning free-text fields, keeping structural categories)
structural_data = graph_data.clone()
desc_masks = torch.zeros(graph_data.num_nodes, dtype=torch.bool)

for cls in [
    'CaseDescription', 'TaskDescription', 'CaseSolutionFinalStatement', 
    'hasModifiedField', 'hasNewValue', 'hasPreviousValue', 
    'hasSalutation', 'hasTaskSubject', 'hasVehicleStructureName'
]:
    desc_masks |= get_graph_mask(graph_data, ontology_obj, schema_label, cls)

mask = ~(desc_masks[graph_data.edge_index[0]] | desc_masks[graph_data.edge_index[1]])
structural_data.edge_index = graph_data.edge_index[:, mask]
structural_data.edge_type = graph_data.edge_type[mask]

pct_removed = ((graph_data.edge_index.size(1) - structural_data.edge_index.size(1)) / graph_data.edge_index.size(1)) * 100.0
log_ablation_run("Structural Core (Metadata Only)", structural_data, pct_removed)

# -- Group C: Minimal Structure
structural_data = graph_data.clone()
desc_masks = torch.zeros(graph_data.num_nodes, dtype=torch.bool)

for cls in [
    'Account', 'AccountType', 'Brand', 'CaseClosureType', 'CaseModification', 
    'CaseReason', 'CaseStatus', 'CaseType', 'Country', 'Language', 
    'PriorityLevel', 'ProductGroup', 'ServiceType', 'SupportLevel', 
    'TaskType', 'TimeInstant', 'Vehicle', 'VehicleClass', 'VehicleSeries', 
    'hasAccountLastModifiedDate', 'hasCaseClosedDate', 'hasCaseConversationAmount', 
    'hasCaseHandlingTimeSec', 'hasCaseSessionAmount', 'hasCreateDate', 
    'hasModificationDate', 'hasTaskCompletedDate', 'hasTaskDurationSec',
    'CaseDescription', 'TaskDescription', 'CaseSolutionFinalStatement', 
    'hasModifiedField', 'hasNewValue', 'hasPreviousValue', 
    'hasSalutation', 'hasTaskSubject', 'hasVehicleStructureName'
]:
    desc_masks |= get_graph_mask(graph_data, ontology_obj, schema_label, cls)

mask = ~(desc_masks[graph_data.edge_index[0]] | desc_masks[graph_data.edge_index[1]])
structural_data.edge_index = graph_data.edge_index[:, mask]
structural_data.edge_type = graph_data.edge_type[mask]

pct_removed = ((graph_data.edge_index.size(1) - structural_data.edge_index.size(1)) / graph_data.edge_index.size(1)) * 100.0
log_ablation_run("Minimal Core", structural_data, pct_removed)

# ── 9. PRINT PEAK PUBLISHABLE REPORT SUMMARY ──────────────────────────────────
df_ablation = pd.DataFrame(ablation_results)
print(f"\n=======================================================")
print(f" ABLATION STUDY COMPLETE FOR {schema_label.upper()}")
print(f"=======================================================")
print(df_ablation[[
    'ablation', 'edges_removed', 'pct_edges_removed', 'nodes_removed', 'pct_nodes_removed', 
    'ROC_AUC', 'delta_ROC_AUC', 'delta_F1', 'RID', 'PR_AUC', 
    'class0_F1', 'class1_F1', 
    'high_signal_text_pct', 'fallback_noise_pct', 
    'runtime_sec', 'best_epoch', 'epochs_run', 'status'
]].to_string(index=False))
print(f"=======================================================")


## K Sweep Test

In [ ]:
# ── EXPERIMENT 2: K-HOP DEPTH SWEEP (RQ1b) ────────────────────────────────────
print("\n>>> STARTING K-SWEEP: STRUCTURAL SENSITIVITY (K = 1, 2, 3)")

# Everything except K is held at the tuned winning config, so any metric
# difference across rows is attributable to depth alone.
def k_sweep_model_factory(hidden_list, bases):
    return RGCNModel(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_relations,
        vector_table=vector_table,
        categorical_row_ids=torch.where(usage_counts > 1)[0],
        num_bases=bases,
    )

# Fan-out fixed at the graph's 99th-percentile neighbourhood size (Table 1)
# and held constant across K=1,2,3, isolating depth from fan-out.
_rq1a_csv_path = f"{volume}graph_geometry/rq1a_structural_comparison.csv"
if not os.path.exists(_rq1a_csv_path):
    raise FileNotFoundError(
        f"Could not find Table 1's structural comparison CSV at: {_rq1a_csv_path}. "
        f"Run extract_graph_geometry.ipynb first -- the K-sweep's fixed fan-out is "
        f"derived from its p99_degree_threshold measurement."
    )
_rq1a_df = pd.read_csv(_rq1a_csv_path, index_col='metric')
_kg_display = {'base': 'Ad-hoc (Base)', 'sulo': 'SULO'}[schema_label]
_p99_fanout = int(round(float(_rq1a_df.loc['p99_degree_threshold', _kg_display])))
print(f">>> Fixed fan-out for {_kg_display}: {_p99_fanout} "
      f"(99th-percentile neighbourhood size, from rq1a_structural_comparison.csv)")

K_SWEEP_VALUES = [
    (1, (_p99_fanout,)),
    (2, (_p99_fanout, _p99_fanout)),
    (3, (_p99_fanout, _p99_fanout, _p99_fanout)),
]

k_sweep_csv_path = f"{volume}rgcn/{schema_label}_k_sweep_results.csv"

k_sweep_df = run_k_sweep(
    model_factory=k_sweep_model_factory,
    graph_data=graph_data, schema_label=schema_label,
    train_indices=train_indices, val_indices=val_indices, test_indices=test_indices,
    vector_table=vector_table, usage_counts=usage_counts,
    hidden_dim=optimal_dim, bases=optimal_bases, lr=optimal_lr, wd=optimal_wd,
    k_values=K_SWEEP_VALUES,
    epochs=50, patience=15,
    csv_path=k_sweep_csv_path,
    resume=True,
)

print(f"\n=======================================================")
print(f" K-SWEEP COMPLETE FOR {schema_label.upper()}")
print(f"=======================================================")
print(k_sweep_df.to_string(index=False))
print(f"=======================================================")

>>> STARTING K-SWEEP: STRUCTURAL SENSITIVITY (K = 1, 2, 3)

[K-SWEEP] K=3 | Neighbors=(15, 10, 5)
  -> CUDA OOM at batch_size=256, retrying at batch_size=128...
Epoch 000 | Train Loss: 0.7606 | Val Loss: 0.6562 | Val Acc: 0.6106 | Val F1: 0.6189 | Val ROC-AUC: 0.6525
  -> Best model saved to RAM! (roc_auc: 0.6525)
Epoch 001 | Train Loss: 0.6588 | Val Loss: 0.6376 | Val Acc: 0.6426 | Val F1: 0.6657 | Val ROC-AUC: 0.6948
  -> Best model saved to RAM! (roc_auc: 0.6948)
Epoch 002 | Train Loss: 0.6461 | Val Loss: 0.6390 | Val Acc: 0.6264 | Val F1: 0.6764 | Val ROC-AUC: 0.7042
  -> Best model saved to RAM! (roc_auc: 0.7042)
Epoch 003 | Train Loss: 0.6385 | Val Loss: 0.6387 | Val Acc: 0.6330 | Val F1: 0.6924 | Val ROC-AUC: 0.7096
  -> Best model saved to RAM! (roc_auc: 0.7096)
Epoch 004 | Train Loss: 0.6355 | Val Loss: 0.6284 | Val Acc: 0.6452 | Val F1: 0.6773 | Val ROC-AUC: 0.7083
Epoch 005 | Train Loss: 0.6302 | Val Loss: 0.6220 | Val Acc: 0.6618 | Val F1: 0.6652 | Val ROC-AUC: 0.7127
  -> 